# Merge house ID + image folder + tabular property data

This notebook combines:
- `satilir_id_price_folder.csv` (house id, price, image folder)
- `satilir_properties.csv` (tabular features + listing link)

The key idea is to derive a comparable slug from:
- `properties.link` (remove `https://arenda.az/`)
- `image_folder_name` (remove numeric prefix like `001_`)

Then merge and export a single dataset.

## 1) Set paths and import libraries

In [1]:
import pandas as pd
from pathlib import Path
import re
import os

BASE_DIR = Path('/home/admin123/AI_project')
ID_PRICE_CSV = BASE_DIR / 'satilir_id_price_folder.csv'
PROPERTIES_CSV = BASE_DIR / 'satilir_properties.csv'
IMAGES_DIR = BASE_DIR / 'satilir_photos'
OUTPUT_CSV = BASE_DIR / 'satilir_merged_with_images.csv'

print('ID/price CSV:', ID_PRICE_CSV)
print('Properties CSV:', PROPERTIES_CSV)
print('Images folder:', IMAGES_DIR)
print('Output CSV:', OUTPUT_CSV)

ID/price CSV: /home/admin123/AI_project/satilir_id_price_folder.csv
Properties CSV: /home/admin123/AI_project/satilir_properties.csv
Images folder: /home/admin123/AI_project/satilir_photos
Output CSV: /home/admin123/AI_project/satilir_merged_with_images.csv


## 2) Load CSV files and standardize column names

In [2]:
def to_snake(col: str) -> str:
    col = col.strip().lower()
    col = re.sub(r'[^a-z0-9]+', '_', col)
    col = re.sub(r'_+', '_', col).strip('_')
    return col

id_price = pd.read_csv(ID_PRICE_CSV)
properties = pd.read_csv(PROPERTIES_CSV)

id_price.columns = [to_snake(c) for c in id_price.columns]
properties.columns = [to_snake(c) for c in properties.columns]

print('id_price shape:', id_price.shape)
print('properties shape:', properties.shape)
print('\nid_price columns:\n', id_price.columns.tolist())
print('\nproperties columns (first 12):\n', properties.columns.tolist()[:12])

print('\nid_price dtypes:\n', id_price.dtypes)
print('\nproperties dtypes (first 12):\n', properties.dtypes.head(12))

id_price shape: (6178, 3)
properties shape: (6183, 27)

id_price columns:
 ['house_id', 'price', 'image_folder_name']

properties columns (first 12):
 ['link', 'price', 'rooms', 'area_m2', 'land_area_sot', 'floor', 'has_document', 'address', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali']

id_price dtypes:
 house_id              int64
price                 int64
image_folder_name    object
dtype: object

properties dtypes (first 12):
 link              object
price              int64
rooms            float64
area_m2          float64
land_area_sot    float64
floor            float64
has_document      object
address           object
avtodayanacaq     object
balkon            object
duzelme           object
esyali            object
dtype: object


## 3) Normalize link and folder keys

In [3]:
def normalize_slug(s: str) -> str:
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    s = s.replace('https://', '').replace('http://', '')
    s = re.sub(r'^www\.', '', s)
    s = re.sub(r'^[^/]+/', '', s)  # remove domain part if present
    s = s.strip('/').strip()
    s = re.sub(r'\s+', '-', s)
    s = re.sub(r'-+', '-', s)
    return s

# Properties: slug from link
properties['link_slug'] = properties['link'].apply(normalize_slug)

# id_price: keep original folder, and comparable slug without numeric prefix
id_price['image_folder_name'] = id_price['image_folder_name'].astype(str).str.strip()
id_price['folder_slug_raw'] = id_price['image_folder_name'].apply(normalize_slug)
id_price['folder_slug_no_id'] = id_price['folder_slug_raw'].str.replace(r'^\d+_', '', regex=True)

print('Sample link_slug:')
print(properties[['link', 'link_slug']].head(3))
print('\nSample folder slugs:')
print(id_price[['image_folder_name', 'folder_slug_raw', 'folder_slug_no_id']].head(3))

Sample link_slug:
                                                link  \
0  https://arenda.az/satilir-3-otaqli-yeni-tikili...   
1  https://arenda.az/satilir-4-otaqli-yeni-tikili...   
2  https://arenda.az/satilir-2-otaqli-yeni-tikili...   

                                           link_slug  
0  satilir-3-otaqli-yeni-tikili-nerimanov-rayonu-...  
1  satilir-4-otaqli-yeni-tikili-nerimanov-rayonu-...  
2  satilir-2-otaqli-yeni-tikili-nerimanov-rayonu-...  

Sample folder slugs:
                                   image_folder_name  \
0  001_satilir-4-otaqli-heyet-evi-villa-abseron-r...   
1  002_satilir-3-otaqli-yeni-tikili-xetai-rayonu-...   
2  003_satilir-2-otaqli-yeni-tikili-yasamal-rayon...   

                                     folder_slug_raw  \
0  001_satilir-4-otaqli-heyet-evi-villa-abseron-r...   
1  002_satilir-3-otaqli-yeni-tikili-xetai-rayonu-...   
2  003_satilir-2-otaqli-yeni-tikili-yasamal-rayon...   

                                   folder_slug_no_id  
0  satilir

## 4) Create ID mapping from image folder names

In [4]:
id_price['house_id_from_folder'] = (
    id_price['image_folder_name']
    .str.extract(r'^(\d+)_', expand=False)
    .astype('Int64')
)

if 'house_id' in id_price.columns:
    id_price['house_id'] = pd.to_numeric(id_price['house_id'], errors='coerce').astype('Int64')
    id_match_rate = (id_price['house_id'] == id_price['house_id_from_folder']).mean()
    print(f'house_id match rate vs folder prefix: {id_match_rate:.2%}')
else:
    id_price['house_id'] = id_price['house_id_from_folder']

id_map = id_price[['house_id', 'price', 'image_folder_name', 'folder_slug_no_id']].copy()
id_map = id_map.rename(columns={'price': 'price_id_source'})

print('id_map shape:', id_map.shape)
print(id_map.head())

house_id match rate vs folder prefix: 100.00%
id_map shape: (6178, 4)
   house_id  price_id_source  \
0         1           115000   
1         2           195500   
2         3            55000   
3         4           200000   
4         5            97000   

                                   image_folder_name  \
0  001_satilir-4-otaqli-heyet-evi-villa-abseron-r...   
1  002_satilir-3-otaqli-yeni-tikili-xetai-rayonu-...   
2  003_satilir-2-otaqli-yeni-tikili-yasamal-rayon...   
3  004_satilir-7-otaqli-bag-evi-bineqedi-rayonu-a...   
4  005_satilir-3-otaqli-yeni-tikili-abseron-rayon...   

                                   folder_slug_no_id  
0  satilir-4-otaqli-heyet-evi-villa-abseron-rayon...  
1  satilir-3-otaqli-yeni-tikili-xetai-rayonu-hezi...  
2  satilir-2-otaqli-yeni-tikili-yasamal-rayonu-in...  
3  satilir-7-otaqli-bag-evi-bineqedi-rayonu-avtov...  
4  satilir-3-otaqli-yeni-tikili-abseron-rayonu-ma...  


## 5) Join price/folder data with property table

In [5]:
merged = properties.merge(
    id_map,
    left_on='link_slug',
    right_on='folder_slug_no_id',
    how='left',
    indicator=True
)

print('Merge indicator counts:')
print(merged['_merge'].value_counts(dropna=False))
print('\nMerged shape:', merged.shape)

Merge indicator counts:
_merge
both          6178
left_only        5
right_only       0
Name: count, dtype: int64

Merged shape: (6183, 33)


## 6) Run data quality checks

In [6]:
# Unmatched rows
unmatched = merged[merged['_merge'] != 'both'].copy()
print('Unmatched rows:', len(unmatched))
if len(unmatched) > 0:
    print(unmatched[['link', 'link_slug']].head(10))

# Duplicate keys in mapping
dup_keys = id_map[id_map['folder_slug_no_id'].duplicated(keep=False)].sort_values('folder_slug_no_id')
print('\nDuplicate folder_slug_no_id in id_map:', dup_keys['folder_slug_no_id'].nunique())
if len(dup_keys) > 0:
    print(dup_keys.head(10))

# Price conflicts (property price vs id_price price)
merged['price'] = pd.to_numeric(merged['price'], errors='coerce')
merged['price_id_source'] = pd.to_numeric(merged['price_id_source'], errors='coerce')
conflict = merged[
    merged['price'].notna() & merged['price_id_source'].notna() & (merged['price'] != merged['price_id_source'])
]
print('\nPrice conflicts:', len(conflict))
if len(conflict) > 0:
    print(conflict[['link', 'price', 'price_id_source', 'house_id', 'image_folder_name']].head(10))

Unmatched rows: 5
                                                   link  \
556   https://arenda.az/satilir-5-otaqli-kohne-tikil...   
2647  https://arenda.az/satilir-3-otaqli-kohne-tikil...   
2649  https://arenda.az/satilir-1-otaqli-kohne-tikil...   
2722  https://arenda.az/satilir-2-otaqli-yeni-tikili...   
4913  https://arenda.az/satilir-4-otaqli-kohne-tikil...   

                                              link_slug  
556   satilir-5-otaqli-kohne-tikili-bineqedi-rayonu-...  
2647  satilir-3-otaqli-kohne-tikili-bineqedi-rayonu-...  
2649  satilir-1-otaqli-kohne-tikili-bineqedi-rayonu-...  
2722  satilir-2-otaqli-yeni-tikili-abseron-rayonu-ma...  
4913  satilir-4-otaqli-kohne-tikili-bineqedi-rayonu-...  

Duplicate folder_slug_no_id in id_map: 0

Price conflicts: 4
                                                   link   price  \
1163  https://arenda.az/satilir-3-otaqli-yeni-tikili...   25000   
1189  https://arenda.az/satilir-2-otaqli-yeni-tikili...   68000   
1194  https://ar

## 7) Build final output table

In [7]:
# choose a final price: prefer id_price source when available, else property price
merged['final_price'] = merged['price_id_source'].combine_first(merged['price'])

# Remove helper columns and keep a clean output layout
helper_cols = {'folder_slug_no_id', 'link_slug', '_merge'}
property_cols = [c for c in properties.columns if c not in helper_cols]

front_cols = ['house_id', 'final_price', 'image_folder_name', 'link']
existing_front_cols = [c for c in front_cols if c in merged.columns]
remaining_cols = [c for c in property_cols if c not in existing_front_cols]

final_df = merged[existing_front_cols + remaining_cols].copy()
final_df = final_df.rename(columns={'final_price': 'price'})

# Avoid duplicate-named columns if any
final_df = final_df.loc[:, ~final_df.columns.duplicated()]

print('Final shape:', final_df.shape)
print('Final columns (first 15):', final_df.columns.tolist()[:15])

Final shape: (6183, 29)
Final columns (first 15): ['house_id', 'price', 'image_folder_name', 'link', 'rooms', 'area_m2', 'land_area_sot', 'floor', 'has_document', 'address', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz']


## 8) Save merged dataset to CSV

In [8]:
final_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f'Saved: {OUTPUT_CSV}')
print(f'Rows: {len(final_df):,}')
print(f'Columns: {len(final_df.columns)}')

display(final_df.head(10))

Saved: /home/admin123/AI_project/satilir_merged_with_images.csv
Rows: 6,183
Columns: 29


,house_id,price,image_folder_name,link,rooms,area_m2,land_area_sot,floor,has_document,address,...,kondisioner,lift,merkezi_qizdirici_sistem,metbex_mebeli,pvc_pencere,qaz,su,telefon,temirli,temirsiz
0,1001,200000.0,1001_satilir-3-otaqli-yeni-tikili-nerimanov-ra...,https://arenda.az/satilir-3-otaqli-yeni-tikili...,3.0,65.0,NaN,5.0,No,"Bakı, Nərimanov, metro 28 May",...,Yes,Yes,No,Yes,Yes,Yes,Yes,No,Yes,No
1,1002,800000.0,1002_satilir-4-otaqli-yeni-tikili-nerimanov-ra...,https://arenda.az/satilir-4-otaqli-yeni-tikili...,4.0,197.0,NaN,9.0,Yes,"Bakı, Nərimanov, metro Gənclik",...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No
2,1003,210000.0,1003_satilir-2-otaqli-yeni-tikili-nerimanov-ra...,https://arenda.az/satilir-2-otaqli-yeni-tikili...,2.0,54.0,NaN,12.0,No,"Bakı, Nərimanov, metro Nəriman Nərimanov",...,No,No,No,No,No,No,No,No,No,No
3,1005,149000.0,1005_satilir-4-otaqli-kohne-tikili-bineqedi-ra...,https://arenda.az/satilir-4-otaqli-kohne-tikil...,4.0,100.0,NaN,2.0,Yes,"Bakı, Binəqədi, Biləcəri qəs.",...,No,No,No,No,No,No,No,No,Yes,No
4,1006,45000.0,1006_satilir-1-otaqli-yeni-tikili-xirdalan-abs...,https://arenda.az/satilir-1-otaqli-yeni-tikili...,1.0,44.0,NaN,1.0,Yes,Xırdalan,...,No,No,No,No,No,No,No,No,No,No
5,1007,109000.0,1007_satilir-3-otaqli-yeni-tikili-xirdalan-h-e...,https://arenda.az/satilir-3-otaqli-yeni-tikili...,3.0,69.0,NaN,13.0,No,Xırdalan,...,No,Yes,No,No,Yes,Yes,Yes,No,Yes,No
6,1008,70500.0,1008_satilir-2-otaqli-yeni-tikili-xirdalan-h-e...,https://arenda.az/satilir-2-otaqli-yeni-tikili...,2.0,73.0,NaN,4.0,No,Xırdalan,...,No,No,No,No,Yes,Yes,Yes,No,Yes,No
7,1009,290000.0,1009_satilir-2-otaqli-yeni-tikili-nesimi-rayon...,https://arenda.az/satilir-2-otaqli-yeni-tikili...,2.0,70.0,NaN,8.0,Yes,"Bakı, Nəsimi",...,Yes,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,No
8,1010,135000.0,1010_satilir-3-otaqli-yeni-tikili-xirdalan-h-a...,https://arenda.az/satilir-3-otaqli-yeni-tikili...,3.0,100.0,NaN,9.0,No,Xırdalan,...,No,Yes,No,Yes,Yes,Yes,Yes,No,Yes,No
9,1011,54000.0,1011_satilir-1-otaqli-yeni-tikili-xirdalan-h-e...,https://arenda.az/satilir-1-otaqli-yeni-tikili...,1.0,31.0,NaN,5.0,No,Xırdalan,...,No,Yes,No,No,Yes,Yes,Yes,No,Yes,No
